# Querying Loan Data with SQL
### BAI709 Data Modelling and Machine Learning

---

**The business question.** A credit manager asks you questions about a table of loan applicants: *How many have a serious delinquency record? What are the biggest loans among the risky ones? Why does the default rate in this table look so low?* Analysts answer questions like these in **SQL**, the language that most company databases speak. You will use it here on a small in-memory database, so there is nothing to install.

---

### How this notebook works

Every part follows five steps.

| Step | What it is |
|---|---|
| 1 | **Load and look** at the table |
| 2 | **Run the query** |
| 3 | **Read the query aloud in plain English** |
| 4 | **Check it against pandas**, which you already know |
| 5 | **Say it to a manager** |

- Run cells in order with **Shift+Enter**. The data is in `../data/`.
- **Copilot is on.** In Part 5 you ask Copilot to write the queries. **Read each query aloud before you run it**, and check the count against pandas. A query you cannot read aloud is one you cannot trust.
- **Your turn** cells are for you to change and re-run. **Stretch** is blank for quick finishers.

## Setup

SQLite is a small database that lives in memory. We copy two tables into it: `home_equity` (3,364 loan applicants) and `hmeq` (5,960 loan applicants). `ask()` runs a query and gives back a table.

In [ ]:
import sqlite3
import pandas as pd

home_equity = pd.read_excel("../data/Home_Equity.xlsx")
hmeq = pd.read_csv("../data/hmeq_full.csv")

db = sqlite3.connect(":memory:")                       # a database that lives only while this notebook runs
home_equity.to_sql("home_equity", db, index=False)
hmeq.to_sql("hmeq", db, index=False)

def ask(query):
    # Run a SQL query and return the answer as a table
    return pd.read_sql_query(query, db)

print("Tables ready: home_equity, hmeq")

> **Concept: a SQL query is a sentence with fixed parts, in a fixed order.**
>
> ```
> SELECT   which columns do you want?
> FROM     which table?
> WHERE    which rows? (can be left out)
> ORDER BY in what order? (can be left out)
> LIMIT    how many rows at most? (can be left out)
> ```
>
> Two **aggregates** summarise many rows into one answer: `COUNT(*)` counts rows and `AVG(column)` averages a column. Because `BAD` is 1 for a default and 0 otherwise, `AVG(BAD)` is the **default rate**.
>
> A blank in the data is stored as **NULL**, which means *unknown*. You test for it with `IS NULL` and `IS NOT NULL`, never with `=`.

---
# Exercise: SELECT, WHERE and ORDER BY

## Part 1: Look at the table

| Step | What it is | What we do in this part |
|---|---|---|
| 1 | **Load and look** | See the first rows and count them. |
| 2 | **Run the query** | Run `SELECT ... FROM ... LIMIT` and `SELECT COUNT(*)`. |
| 3 | **Read the query aloud in plain English** | Read each query aloud. |
| 4 | **Check against pandas** | Compare the count with `len()` in pandas. |
| 5 | **Say it to a manager** | Say how big the table is. |

### Steps 1 and 2: Load, look and run

In [ ]:
ask("SELECT * FROM home_equity LIMIT 5")

In [ ]:
ask("SELECT COUNT(*) FROM home_equity")

### Step 3: Read the queries aloud

- *"Select everything from the home_equity table, but only the first five rows."*
- *"Select a count of the rows in the home_equity table."*

### Step 4: Check against pandas

In [ ]:
print(f"pandas says the table has {len(home_equity):,} rows.")

### Step 5: Say it to a manager

*One sentence:* how many applicants are in the table?

*Your sentence here*

---
## Part 2: WHERE picks the rows

| Step | What it is | What we do in this part |
|---|---|---|
| 1 | **Load and look** | Use the same table. |
| 2 | **Run the query** | Count the applicants with three or more delinquent credit lines and find their default rate. |
| 3 | **Read the query aloud in plain English** | Read the query aloud. |
| 4 | **Check against pandas** | Reproduce the count and the rate with a pandas filter. |
| 5 | **Say it to a manager** | Say what a manager should take from it. |

In [ ]:
ask("SELECT COUNT(*), AVG(BAD) FROM home_equity WHERE DELINQ >= 3")

### Step 3: Read it aloud

*"Count the rows, and average BAD, from the home_equity table, but only for applicants with three or more delinquent credit lines."*

### Step 4: Check against pandas

In [ ]:
subset = home_equity[home_equity["DELINQ"] >= 3]
print(f"pandas: {len(subset)} applicants, default rate {subset['BAD'].mean():.3f}")

### Your turn

Change the threshold from 3 to another number and re-run both the query and the pandas check. Do the two always agree?

In [ ]:
LINES = 3            # <- change me
sql_answer = ask(f"SELECT COUNT(*), AVG(BAD) FROM home_equity WHERE DELINQ >= {LINES}")
pandas_answer = home_equity[home_equity["DELINQ"] >= LINES]["BAD"].agg(["size", "mean"])
print(sql_answer.values.round(3), pandas_answer.values.round(3))

### Step 5: Say it to a manager

*Two sentences:* what share of applicants with a poor delinquency record default, and how does that compare with the whole table?

*Your two sentences here*

---
## Part 3: ORDER BY and LIMIT

| Step | What it is | What we do in this part |
|---|---|---|
| 1 | **Load and look** | Use the same table. |
| 2 | **Run the query** | List the five largest loans among applicants who defaulted and had at least three delinquent lines. |
| 3 | **Read the query aloud in plain English** | Read the query aloud. |
| 4 | **Check against pandas** | Reproduce the list with pandas `sort_values`. |
| 5 | **Say it to a manager** | Say what the list shows. |

In [ ]:
ask("SELECT LOAN, VALUE, JOB, DELINQ, BAD FROM home_equity WHERE BAD = 1 AND DELINQ >= 3 ORDER BY LOAN DESC LIMIT 5")

### Step 3: Read it aloud

*"Show the loan, value, job, delinquencies and outcome, from the home_equity table, only for applicants who defaulted **and** had three or more delinquent lines, with the biggest loan first, and stop after five."*

### Step 4: Check against pandas

In [ ]:
check = home_equity[(home_equity["BAD"] == 1) & (home_equity["DELINQ"] >= 3)]
check.sort_values("LOAN", ascending=False)[["LOAN", "VALUE", "JOB", "DELINQ", "BAD"]].head(5)

### A check that fails quietly

A colleague wants the **five largest** loans among the same group. The query below runs without an error. Look at the loans it returns. Are they the largest? Fix it.

In [ ]:
ask("SELECT LOAN, VALUE, JOB, DELINQ, BAD FROM home_equity WHERE BAD = 1 AND DELINQ >= 3 ORDER BY LOAN LIMIT 5")

### Step 5: Say it to a manager

*One sentence:* what does the list of the five largest loans tell the credit manager?

*Your sentence here*

---
## Part 4: Why does this table look so safe?

| Step | What it is | What we do in this part |
|---|---|---|
| 1 | **Load and look** | Compare `home_equity` with the full `hmeq` table. |
| 2 | **Run the query** | Count and average with `IS NULL` and `IS NOT NULL`. |
| 3 | **Read the query aloud in plain English** | Read what the blank figures mean. |
| 4 | **Check against pandas** | Check by counting the blanks in pandas. |
| 5 | **Say it to a manager** | Say what the missing rows hid. |

The default rate in `home_equity` is far lower than in the real file. The reason is in how `home_equity` was made: it holds **only the applicants with no gaps in any column**, and the rows with gaps were the risky ones.

### Step 2: Run the queries

In [ ]:
print("Default rate in each table:")
print(ask("SELECT COUNT(*) AS applicants, AVG(BAD) AS default_rate FROM home_equity").to_string(index=False))
print(ask("SELECT COUNT(*) AS applicants, AVG(BAD) AS default_rate FROM hmeq").to_string(index=False))

In [ ]:
print("Applicants with NO debt-to-income figure (hmeq):")
print(ask("SELECT COUNT(*) AS applicants, AVG(BAD) AS default_rate FROM hmeq WHERE DEBTINC IS NULL").to_string(index=False))
print("\nApplicants WITH a debt-to-income figure (hmeq):")
print(ask("SELECT COUNT(*) AS applicants, AVG(BAD) AS default_rate FROM hmeq WHERE DEBTINC IS NOT NULL").to_string(index=False))
print("\nThe same question asked of home_equity:")
print(ask("SELECT COUNT(*) AS applicants FROM home_equity WHERE DEBTINC IS NULL").to_string(index=False))

### Step 3: Read it in plain English

In [ ]:
a = ask("SELECT COUNT(*) AS n, AVG(BAD) AS r FROM hmeq WHERE DEBTINC IS NULL").iloc[0]
b = ask("SELECT COUNT(*) AS n, AVG(BAD) AS r FROM hmeq WHERE DEBTINC IS NOT NULL").iloc[0]
he = ask("SELECT AVG(BAD) FROM home_equity").iloc[0, 0]
full_rate = ask("SELECT AVG(BAD) FROM hmeq").iloc[0, 0]
print(f"{int(a['n']):,} applicants have no debt-to-income figure, and {a['r']:.1%} of them default. Of the {int(b['n']):,} who have one, {b['r']:.1%} default.")
print(f"home_equity keeps only complete rows, so it contains none of the blank-figure applicants, and its default rate is {he:.1%} against {full_rate:.1%} in the full table.")
print("Dropping every row with a gap made the lender look about half as risky as it is, because the rows with gaps were the risky ones.")

### Two queries, one question

The manager asks: *"How many applicants have no debt-to-income figure?"* Two analysts write different queries. Which one is right? Predict first, then run.

In [ ]:
query_a = "SELECT COUNT(*) FROM hmeq WHERE DEBTINC IS NULL"
query_b = "SELECT COUNT(*) FROM hmeq WHERE DEBTINC = NULL"
print("A:", query_a, "->", ask(query_a).iloc[0, 0])
print("B:", query_b, "->", ask(query_b).iloc[0, 0])

NULL means *unknown*, and "is an unknown value equal to an unknown value?" has no answer, so `= NULL` matches nothing and quietly returns 0. Always use `IS NULL`.

### Step 4: Check against pandas

In [ ]:
print(f"pandas: {hmeq['DEBTINC'].isna().sum():,} blanks in DEBTINC; default rate {hmeq.loc[hmeq['DEBTINC'].isna(), 'BAD'].mean():.3f}")

### Step 5: Say it to a manager

*Two sentences:* what did deleting the incomplete rows hide, and what should the lender do about the missing figure?

*Your two sentences here*

---
## Part 5: Let Copilot write the query

| Step | What it is | What we do in this part |
|---|---|---|
| 1 | **Load and look** | Use the `hmeq` table, which has all the applicants. |
| 2 | **Run the query** | Ask Copilot for one query per business question. |
| 3 | **Read the query aloud in plain English** | Read every query aloud before you run it. |
| 4 | **Check against pandas** | Check each answer against a pandas calculation. |
| 5 | **Say it to a manager** | Say what you would tell the manager. |

Three questions from the credit manager. For each one, **write a prompt to Copilot** that names the table (`hmeq`) and its columns (`BAD`, `LOAN`, `JOB`, `DEBTINC`, `DELINQ`), paste the query it gives you into the cell, **read it aloud**, and run it.

1. How many applicants asked for a loan of more than 30,000, and what share of them defaulted?
2. What are the five largest loans among applicants whose job is Sales? Show the loan, the job and the outcome.
3. How many applicants have no job recorded, and what share of them defaulted?

> *Example prompt:* "Write a SQLite query on a table called hmeq with columns BAD (1 = default), LOAN and JOB. Count the applicants with LOAN above 30000 and give the average of BAD."

In [ ]:
def run(query):
    # Runs a pasted query, or reminds you to paste one
    if query.strip():
        return ask(query)
    print("Paste Copilot's query into the quotes above, then run this cell again.")

QUERY_1 = ""     # paste Copilot's query for question 1
run(QUERY_1)

In [ ]:
QUERY_2 = ""     # paste Copilot's query for question 2
run(QUERY_2)

In [ ]:
QUERY_3 = ""     # paste Copilot's query for question 3
run(QUERY_3)

### Step 4: Check each answer against pandas

If your query's answer differs from these, read the query aloud again: something in the wording asked a different question. Two Sales loans tie at 28,900, so those two rows may come out in either order. That is not an error.

In [ ]:
big = hmeq[hmeq["LOAN"] > 30000]
print(f"1. {len(big):,} applicants with a loan above 30,000; default rate {big['BAD'].mean():.3f}")
sales = hmeq[hmeq["JOB"] == "Sales"].sort_values("LOAN", ascending=False)[["LOAN", "JOB", "BAD"]].head(5)
print("2. Five largest loans, Sales:")
print(sales.to_string())
nojob = hmeq[hmeq["JOB"].isna()]
print(f"3. {len(nojob):,} applicants with no job recorded; default rate {nojob['BAD'].mean():.3f}")

### Step 5: Say it to a manager (fixed cell)

Two sentences in business language: what did the queries show about the big loans and about the applicants with no job recorded?

*Your two sentences here*

### Talk it through

1. Copilot wrote a query that ran and gave a plausible number, but the wording of your prompt had asked a slightly different question. How would you catch that?
2. When is SQL the right tool and when is pandas? Think about where the data lives and how big it is.

---
## Stretch

Blank on purpose. Ask Copilot for a query of your own on `hmeq` (for example, applicants with a high debt-to-income figure and a recent inquiry), read it aloud, and check it against pandas.

In [ ]:
# Stretch: your query here

---
*BAI709 Data Modelling and Machine Learning, Regent's University London*